# Milestone 1: CPU reference pipeline (C++)

Run every cell from top to bottom, waiting for each to finish. Only the two cells marked **CHECKPOINT** need you to look at the output.

This milestone does **not** use the GPU, but keep the T4 runtime selected so the session stays the same for later milestones.

## Step 1: Connect Drive and go to the project folder
Run this at the start of **every** session. If it asks for permission, click Allow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project
!ls data

## Step 2: Write the source files
Each cell below saves one file into your Drive project folder (it prints `Writing ...` or `Overwriting ...`). Just run them.

How the code is split:
- `isp_io.h` / `isp_io.cpp`: reading `bayer.bin` and `metadata.txt`, writing the output. Reused later by the CUDA programs.
- `cpu_stages.h`: the list of the 5 stage functions.
- `cpu_pointwise_stages.cpp`: stages 1, 3, 5. Each output pixel depends only on the same input pixel.
- `cpu_neighbourhood_stages.cpp`: stages 2 and 4. Each output pixel needs its 3x3 neighbours.
- `cpu_main.cpp`: loads data, runs and times each stage, saves output.
- `view_output.py`: converts the output to a PNG.

In [ ]:
%%writefile isp_io.h
// isp_io.h
// Reading and writing the simple files our pipeline uses.
// Shared by the CPU program now and the CUDA programs later.

// "#pragma once" tells the compiler: if this file is included twice,
// only use it the first time. It prevents "defined twice" errors.
#pragma once

#include <cstdint>
#include <string>
#include <vector>

// All the numbers from metadata.txt, kept together in one place
// so we can pass them to functions as a single argument.
struct ImageMetadata {
    int imageWidth;
    int imageHeight;
    float blackLevel;
    float whiteLevel;
    float whiteBalanceRed;
    float whiteBalanceGreen;
    float whiteBalanceBlue;
};

// Each function returns true on success and false on failure,
// so main() can stop with a clear message instead of crashing.
bool readMetadataFile(const std::string& metadataPath, ImageMetadata& metadata);

bool readBayerFile(const std::string& bayerPath, int pixelCount,
                   std::vector<uint16_t>& bayerPixels);

bool writeRgbFile(const std::string& outputPath,
                  const std::vector<unsigned char>& rgbBytes);


In [ ]:
%%writefile isp_io.cpp
// isp_io.cpp
// File reading and writing. No image processing happens here.

#include "isp_io.h"

#include <fstream>
#include <iostream>

bool readMetadataFile(const std::string& metadataPath, ImageMetadata& metadata) {
    std::ifstream metadataFile(metadataPath);
    if (!metadataFile.is_open()) {
        std::cout << "ERROR: could not open " << metadataPath << std::endl;
        return false;
    }

    // The file has lines like "width 5796". We read a name, then a number,
    // until the file runs out. We count fields so we can detect a broken file.
    int fieldsFound = 0;
    std::string fieldName;
    float fieldValue = 0.0f;
    while (true) {
        metadataFile >> fieldName;
        metadataFile >> fieldValue;
        if (metadataFile.fail()) {
            break;
        }

        if (fieldName == "width") {
            metadata.imageWidth = static_cast<int>(fieldValue);
            fieldsFound = fieldsFound + 1;
        } else if (fieldName == "height") {
            metadata.imageHeight = static_cast<int>(fieldValue);
            fieldsFound = fieldsFound + 1;
        } else if (fieldName == "black_level") {
            metadata.blackLevel = fieldValue;
            fieldsFound = fieldsFound + 1;
        } else if (fieldName == "white_level") {
            metadata.whiteLevel = fieldValue;
            fieldsFound = fieldsFound + 1;
        } else if (fieldName == "wb_red") {
            metadata.whiteBalanceRed = fieldValue;
            fieldsFound = fieldsFound + 1;
        } else if (fieldName == "wb_green") {
            metadata.whiteBalanceGreen = fieldValue;
            fieldsFound = fieldsFound + 1;
        } else if (fieldName == "wb_blue") {
            metadata.whiteBalanceBlue = fieldValue;
            fieldsFound = fieldsFound + 1;
        }
    }

    if (fieldsFound != 7) {
        std::cout << "ERROR: metadata file has " << fieldsFound
                  << " of the 7 expected fields" << std::endl;
        return false;
    }
    return true;
}

bool readBayerFile(const std::string& bayerPath, int pixelCount,
                   std::vector<uint16_t>& bayerPixels) {
    // std::ios::binary means "give me the raw bytes, do not treat this as text".
    std::ifstream bayerFile(bayerPath, std::ios::binary);
    if (!bayerFile.is_open()) {
        std::cout << "ERROR: could not open " << bayerPath << std::endl;
        return false;
    }

    // Make room for every pixel before reading into the vector.
    bayerPixels.resize(pixelCount);

    // Python saved little-endian 16-bit numbers. The CPUs we use (Colab's x86)
    // are also little-endian, so the bytes can be copied straight into memory.
    // read() wants a char pointer, so we reinterpret our uint16_t array as bytes.
    long long expectedByteCount = static_cast<long long>(pixelCount) * 2;
    char* destinationBytes = reinterpret_cast<char*>(bayerPixels.data());
    bayerFile.read(destinationBytes, expectedByteCount);

    long long actualByteCount = bayerFile.gcount();
    if (actualByteCount != expectedByteCount) {
        std::cout << "ERROR: expected " << expectedByteCount << " bytes but read "
                  << actualByteCount << std::endl;
        return false;
    }
    return true;
}

bool writeRgbFile(const std::string& outputPath,
                  const std::vector<unsigned char>& rgbBytes) {
    std::ofstream outputFile(outputPath, std::ios::binary);
    if (!outputFile.is_open()) {
        std::cout << "ERROR: could not create " << outputPath << std::endl;
        return false;
    }

    // The output is just R,G,B bytes for each pixel, row by row. Python reshapes it.
    const char* sourceBytes = reinterpret_cast<const char*>(rgbBytes.data());
    long long byteCount = static_cast<long long>(rgbBytes.size());
    outputFile.write(sourceBytes, byteCount);
    return true;
}


In [ ]:
%%writefile cpu_stages.h
// cpu_stages.h
// The five ISP stages, CPU version. One function per stage.
//
// Image layouts used between stages:
//   Bayer image: one float per pixel, index = row * imageWidth + column.
//   RGB image:   three floats per pixel (red, green, blue next to each other),
//                index = (row * imageWidth + column) * 3 + channel,
//                where channel is 0 for red, 1 for green, 2 for blue.

#pragma once

#include <cstdint>
#include <vector>

#include "isp_io.h"

// Stage 1 (cpu_pointwise_stages.cpp)
void blackLevelCorrection(const std::vector<uint16_t>& bayerPixels,
                          const ImageMetadata& metadata,
                          std::vector<float>& normalisedBayer);

// Stage 2 (cpu_neighbourhood_stages.cpp)
void demosaicBilinear(const std::vector<float>& normalisedBayer,
                      int imageWidth, int imageHeight,
                      std::vector<float>& rgbImage);

// Stage 3 (cpu_pointwise_stages.cpp) - changes rgbImage in place.
void whiteBalance(std::vector<float>& rgbImage, const ImageMetadata& metadata);

// Stage 4 (cpu_neighbourhood_stages.cpp)
void denoiseGaussian3x3(const std::vector<float>& rgbImage,
                        int imageWidth, int imageHeight,
                        std::vector<float>& denoisedImage);

// Stage 5 (cpu_pointwise_stages.cpp)
void gammaAndConvertTo8Bit(const std::vector<float>& rgbImage,
                           std::vector<unsigned char>& outputBytes);


In [ ]:
%%writefile cpu_pointwise_stages.cpp
// cpu_pointwise_stages.cpp
// Stages 1, 3 and 5. These are "pointwise": each output value depends only on
// the SAME pixel of the input, never on its neighbours. That makes them the
// easiest stages to move to the GPU later (one thread per pixel, no sharing).

#include "cpu_stages.h"

#include <cmath>

// Stage 1: black level correction.
// The sensor never reports 0 in darkness; it reports about blackLevel.
// Subtracting it makes "no light" equal 0. Dividing by (white - black) makes
// "fully saturated" equal 1. After this, every value is in 0.0 to 1.0.
void blackLevelCorrection(const std::vector<uint16_t>& bayerPixels,
                          const ImageMetadata& metadata,
                          std::vector<float>& normalisedBayer) {
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    normalisedBayer.resize(pixelCount);

    float usefulRange = metadata.whiteLevel - metadata.blackLevel;

    for (int pixelIndex = 0; pixelIndex < pixelCount; pixelIndex++) {
        float rawValue = static_cast<float>(bayerPixels[pixelIndex]);
        float normalisedValue = (rawValue - metadata.blackLevel) / usefulRange;

        // Noise can push dark pixels slightly below black (your file has 2023 < 2047),
        // and blown highlights can be above white (15464 > 15000). Clamp both.
        if (normalisedValue < 0.0f) {
            normalisedValue = 0.0f;
        }
        if (normalisedValue > 1.0f) {
            normalisedValue = 1.0f;
        }

        normalisedBayer[pixelIndex] = normalisedValue;
    }
}

// Stage 3: white balance.
// The sensor is more sensitive to green than to red or blue, so a white wall
// comes out greenish. Multiplying red and blue by the camera's gains makes
// white objects have equal red, green and blue again.
void whiteBalance(std::vector<float>& rgbImage, const ImageMetadata& metadata) {
    int pixelCount = static_cast<int>(rgbImage.size() / 3);

    for (int pixelIndex = 0; pixelIndex < pixelCount; pixelIndex++) {
        int redIndex = pixelIndex * 3 + 0;
        int greenIndex = pixelIndex * 3 + 1;
        int blueIndex = pixelIndex * 3 + 2;

        float redValue = rgbImage[redIndex] * metadata.whiteBalanceRed;
        float greenValue = rgbImage[greenIndex] * metadata.whiteBalanceGreen;
        float blueValue = rgbImage[blueIndex] * metadata.whiteBalanceBlue;

        // A gain above 1 can push bright values past 1.0. Clamp so saturated
        // areas become white instead of an out-of-range colour.
        if (redValue > 1.0f) {
            redValue = 1.0f;
        }
        if (greenValue > 1.0f) {
            greenValue = 1.0f;
        }
        if (blueValue > 1.0f) {
            blueValue = 1.0f;
        }

        rgbImage[redIndex] = redValue;
        rgbImage[greenIndex] = greenValue;
        rgbImage[blueIndex] = blueValue;
    }
}

// Stage 5: gamma correction and conversion to 8-bit.
// Sensor values are "linear" (twice the light = twice the number), but screens
// and our eyes expect a curve. Raising to the power 1/2.2 brightens the dark
// tones so the photo looks natural. Then we scale 0..1 up to 0..255.
void gammaAndConvertTo8Bit(const std::vector<float>& rgbImage,
                           std::vector<unsigned char>& outputBytes) {
    int valueCount = static_cast<int>(rgbImage.size());
    outputBytes.resize(valueCount);

    float gammaExponent = 1.0f / 2.2f;

    for (int valueIndex = 0; valueIndex < valueCount; valueIndex++) {
        float linearValue = rgbImage[valueIndex];
        if (linearValue < 0.0f) {
            linearValue = 0.0f;
        }
        if (linearValue > 1.0f) {
            linearValue = 1.0f;
        }

        float gammaValue = std::pow(linearValue, gammaExponent);

        // Adding 0.5 before converting rounds to the nearest whole number
        // instead of always rounding down.
        float scaledValue = gammaValue * 255.0f + 0.5f;
        outputBytes[valueIndex] = static_cast<unsigned char>(scaledValue);
    }
}


In [ ]:
%%writefile cpu_neighbourhood_stages.cpp
// cpu_neighbourhood_stages.cpp
// Stages 2 and 4. These are "neighbourhood" stages: each output pixel needs
// the 3x3 block of input pixels around it. They read 9 values per pixel, which
// is why they will be the most interesting stages to optimise on the GPU.

#include "cpu_stages.h"

// Border handling for the Bayer image: "reflect".
// A pixel on the edge has missing neighbours outside the image. For the Bayer
// image we must replace them with a pixel of the SAME COLOUR. Reflecting by one
// (index -1 becomes 1, index size becomes size-2) moves by 2 pixels, and since
// the Bayer pattern repeats every 2 pixels, the colour stays the same.
// (Clamping -1 to 0 would give a pixel of a DIFFERENT colour - wrong.)
static int reflectIndex(int index, int size) {
    if (index < 0) {
        return -index;
    }
    if (index >= size) {
        return 2 * size - 2 - index;
    }
    return index;
}

// Border handling for the RGB image: "clamp" (use the nearest edge pixel).
// After demosaic every pixel has all three colours, so any neighbour works.
static int clampIndex(int index, int size) {
    if (index < 0) {
        return 0;
    }
    if (index >= size) {
        return size - 1;
    }
    return index;
}

static float readBayerValue(const std::vector<float>& bayerImage,
                            int imageWidth, int imageHeight, int row, int column) {
    int safeRow = reflectIndex(row, imageHeight);
    int safeColumn = reflectIndex(column, imageWidth);
    int pixelIndex = safeRow * imageWidth + safeColumn;
    return bayerImage[pixelIndex];
}

// Stage 2: bilinear demosaic for an RGGB sensor.
// The 2x2 tile is      R G      (even row, even column) = red
//                      G B      (odd row, odd column)   = blue
// Every pixel knows one colour. The two missing colours are the average of the
// nearest neighbours that DO know that colour:
//   at red:    green = average of up/down/left/right, blue = average of 4 diagonals
//   at blue:   green = average of up/down/left/right, red  = average of 4 diagonals
//   at green on a red row:  red = average of left/right, blue = average of up/down
//   at green on a blue row: red = average of up/down,  blue = average of left/right
void demosaicBilinear(const std::vector<float>& normalisedBayer,
                      int imageWidth, int imageHeight,
                      std::vector<float>& rgbImage) {
    rgbImage.resize(static_cast<size_t>(imageWidth) * imageHeight * 3);

    for (int row = 0; row < imageHeight; row++) {
        for (int column = 0; column < imageWidth; column++) {
            // Read the full 3x3 neighbourhood once. Not every value is used for
            // every pixel colour, but reading all 9 keeps the code uniform.
            float centre = readBayerValue(normalisedBayer, imageWidth, imageHeight, row, column);
            float up = readBayerValue(normalisedBayer, imageWidth, imageHeight, row - 1, column);
            float down = readBayerValue(normalisedBayer, imageWidth, imageHeight, row + 1, column);
            float left = readBayerValue(normalisedBayer, imageWidth, imageHeight, row, column - 1);
            float right = readBayerValue(normalisedBayer, imageWidth, imageHeight, row, column + 1);
            float upLeft = readBayerValue(normalisedBayer, imageWidth, imageHeight, row - 1, column - 1);
            float upRight = readBayerValue(normalisedBayer, imageWidth, imageHeight, row - 1, column + 1);
            float downLeft = readBayerValue(normalisedBayer, imageWidth, imageHeight, row + 1, column - 1);
            float downRight = readBayerValue(normalisedBayer, imageWidth, imageHeight, row + 1, column + 1);

            float crossAverage = (up + down + left + right) / 4.0f;
            float diagonalAverage = (upLeft + upRight + downLeft + downRight) / 4.0f;
            float horizontalAverage = (left + right) / 2.0f;
            float verticalAverage = (up + down) / 2.0f;

            bool isEvenRow = (row % 2 == 0);
            bool isEvenColumn = (column % 2 == 0);

            float redValue = 0.0f;
            float greenValue = 0.0f;
            float blueValue = 0.0f;

            if (isEvenRow && isEvenColumn) {
                // Red pixel.
                redValue = centre;
                greenValue = crossAverage;
                blueValue = diagonalAverage;
            } else if (isEvenRow && !isEvenColumn) {
                // Green pixel on a red row: red is left/right, blue is up/down.
                redValue = horizontalAverage;
                greenValue = centre;
                blueValue = verticalAverage;
            } else if (!isEvenRow && isEvenColumn) {
                // Green pixel on a blue row: red is up/down, blue is left/right.
                redValue = verticalAverage;
                greenValue = centre;
                blueValue = horizontalAverage;
            } else {
                // Blue pixel.
                redValue = diagonalAverage;
                greenValue = crossAverage;
                blueValue = centre;
            }

            size_t pixelIndex = static_cast<size_t>(row) * imageWidth + column;
            rgbImage[pixelIndex * 3 + 0] = redValue;
            rgbImage[pixelIndex * 3 + 1] = greenValue;
            rgbImage[pixelIndex * 3 + 2] = blueValue;
        }
    }
}

// Stage 4: 3x3 Gaussian blur on each colour channel.
// Weights:   1 2 1
//            2 4 2     divided by 16 (the sum), so brightness is unchanged.
//            1 2 1
// The centre counts most, close neighbours less. Random noise differs from
// pixel to pixel, so averaging cancels some of it while edges stay fairly sharp.
// The output MUST be a separate buffer: if we wrote into the input, later
// pixels would average already-blurred values.
void denoiseGaussian3x3(const std::vector<float>& rgbImage,
                        int imageWidth, int imageHeight,
                        std::vector<float>& denoisedImage) {
    denoisedImage.resize(rgbImage.size());

    const float gaussianWeights[3][3] = {
        {1.0f, 2.0f, 1.0f},
        {2.0f, 4.0f, 2.0f},
        {1.0f, 2.0f, 1.0f}
    };
    const float weightSum = 16.0f;

    for (int row = 0; row < imageHeight; row++) {
        for (int column = 0; column < imageWidth; column++) {
            for (int channel = 0; channel < 3; channel++) {
                float weightedTotal = 0.0f;

                for (int rowOffset = -1; rowOffset <= 1; rowOffset++) {
                    for (int columnOffset = -1; columnOffset <= 1; columnOffset++) {
                        int neighbourRow = clampIndex(row + rowOffset, imageHeight);
                        int neighbourColumn = clampIndex(column + columnOffset, imageWidth);
                        size_t neighbourPixel = static_cast<size_t>(neighbourRow) * imageWidth + neighbourColumn;
                        float neighbourValue = rgbImage[neighbourPixel * 3 + channel];
                        float weight = gaussianWeights[rowOffset + 1][columnOffset + 1];
                        weightedTotal = weightedTotal + neighbourValue * weight;
                    }
                }

                size_t pixelIndex = static_cast<size_t>(row) * imageWidth + column;
                denoisedImage[pixelIndex * 3 + channel] = weightedTotal / weightSum;
            }
        }
    }
}


In [ ]:
%%writefile cpu_main.cpp
// cpu_main.cpp
// CPU reference pipeline. Reads data/bayer.bin, runs the 5 stages, times each
// one, and writes output/cpu_output.bin (8-bit RGB). Every GPU version later
// will be checked against this output.

#include <chrono>
#include <iostream>
#include <vector>

#include "cpu_stages.h"
#include "isp_io.h"

// A shorter name for the clock's time type, so the timing lines stay readable.
// steady_clock is used because it never jumps (unlike the wall clock).
using TimePoint = std::chrono::steady_clock::time_point;

static TimePoint currentTime() {
    return std::chrono::steady_clock::now();
}

static double millisecondsBetween(TimePoint startTime, TimePoint endTime) {
    std::chrono::duration<double, std::milli> elapsed = endTime - startTime;
    return elapsed.count();
}

int main() {
    // ---- Load input (not timed: file reading is not part of the pipeline) ----
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }

    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;
    int pixelCount = imageWidth * imageHeight;
    std::cout << "Image: " << imageWidth << " x " << imageHeight
              << " (" << pixelCount << " pixels)" << std::endl;

    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    // ---- Run and time each stage ----
    std::vector<float> normalisedBayer;
    std::vector<float> rgbImage;
    std::vector<float> denoisedImage;
    std::vector<unsigned char> outputBytes;

    TimePoint pipelineStart = currentTime();

    TimePoint stageStart = currentTime();
    blackLevelCorrection(bayerPixels, metadata, normalisedBayer);
    double blackLevelMs = millisecondsBetween(stageStart, currentTime());

    stageStart = currentTime();
    demosaicBilinear(normalisedBayer, imageWidth, imageHeight, rgbImage);
    double demosaicMs = millisecondsBetween(stageStart, currentTime());

    stageStart = currentTime();
    whiteBalance(rgbImage, metadata);
    double whiteBalanceMs = millisecondsBetween(stageStart, currentTime());

    stageStart = currentTime();
    denoiseGaussian3x3(rgbImage, imageWidth, imageHeight, denoisedImage);
    double denoiseMs = millisecondsBetween(stageStart, currentTime());

    stageStart = currentTime();
    gammaAndConvertTo8Bit(denoisedImage, outputBytes);
    double gammaMs = millisecondsBetween(stageStart, currentTime());

    double totalMs = millisecondsBetween(pipelineStart, currentTime());

    std::cout << "Stage 1 black level : " << blackLevelMs << " ms" << std::endl;
    std::cout << "Stage 2 demosaic    : " << demosaicMs << " ms" << std::endl;
    std::cout << "Stage 3 white bal.  : " << whiteBalanceMs << " ms" << std::endl;
    std::cout << "Stage 4 denoise     : " << denoiseMs << " ms" << std::endl;
    std::cout << "Stage 5 gamma       : " << gammaMs << " ms" << std::endl;
    std::cout << "Total pipeline      : " << totalMs << " ms" << std::endl;

    // ---- Save output (not timed) ----
    bool writeOk = writeRgbFile("output/cpu_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    std::cout << "Saved output/cpu_output.bin" << std::endl;
    return 0;
}


In [ ]:
%%writefile view_output.py
# view_output.py
# Usage: python view_output.py output/cpu_output.bin output/cpu_output.png
# Turns the raw 8-bit RGB file from C++/CUDA into a PNG you can look at.

import sys
import numpy
from PIL import Image

inputPath = sys.argv[1]
outputPath = sys.argv[2]

# Read width and height from the same metadata file the C++ program used.
metadata = {}
for line in open('data/metadata.txt'):
    parts = line.split()
    metadata[parts[0]] = float(parts[1])
imageWidth = int(metadata['width'])
imageHeight = int(metadata['height'])

rgbBytes = numpy.fromfile(inputPath, dtype=numpy.uint8)
expectedSize = imageWidth * imageHeight * 3
if rgbBytes.size != expectedSize:
    print('ERROR: file has', rgbBytes.size, 'bytes, expected', expectedSize)
    sys.exit(1)

rgbImage = rgbBytes.reshape(imageHeight, imageWidth, 3)
Image.fromarray(rgbImage).save(outputPath)

# Same 200x200 centre crop as the Bayer zoom in Milestone 0, for comparison.
centreRow = imageHeight // 2
centreColumn = imageWidth // 2
zoomCrop = rgbImage[centreRow:centreRow + 200, centreColumn:centreColumn + 200]
zoomPath = outputPath.replace('.png', '_zoom.png')
Image.fromarray(zoomCrop).resize((800, 800), Image.NEAREST).save(zoomPath)

print('Saved', outputPath, 'and', zoomPath)


## Step 3: Compile
`g++` is the C++ compiler. `-O2` turns on the normal optimisations, so the CPU timing is fair when we compare it to the GPU later. `-Wall -Wextra` asks for warnings about suspicious code.

The program is saved to `/content/cpu_isp` (Colab's local disk), not to Drive, because Google Drive does not reliably allow running programs stored on it. It is rebuilt in seconds whenever needed.

**Expected:** no output at all. Silence means success.

In [ ]:
!g++ -O2 -std=c++17 -Wall -Wextra -o /content/cpu_isp cpu_main.cpp cpu_pointwise_stages.cpp cpu_neighbourhood_stages.cpp isp_io.cpp

## Step 4: Run (CHECKPOINT 1)
**Expected:** `Image: 5796 x 3870 (22430520 pixels)`, then a time in milliseconds for each stage and the total, then `Saved output/cpu_output.bin`. Demosaic and denoise should be the slowest stages, because they read 9 neighbours per pixel. It takes a few seconds.

**Copy the whole output and paste it to me.** These are your CPU baseline numbers.

In [ ]:
!/content/cpu_isp

## Step 5: View the result (CHECKPOINT 2)
**Expected:** a normal colour photo of the telephone. The colours may look a bit flat or slightly off, because we skip the colour correction matrix. That is fine.

The zoomed crop is the same spot as the Milestone 0 zoom. The dot-matrix pattern should be **gone**, replaced by smooth colour. That proves demosaicing worked.

In [ ]:
!python view_output.py output/cpu_output.bin output/cpu_output.png
from PIL import Image
from IPython.display import display
fullImage = Image.open('output/cpu_output.png')
fullImage.thumbnail((1000, 1000))
display(fullImage)
display(Image.open('output/cpu_output_zoom.png'))